<a href="https://colab.research.google.com/github/stellaephile/netascore_india/blob/main/extract_geofabrik_osm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install osmium-tool in Google Colab

# print("Installing osmium-tool...")
!apt-get update -qq
!apt-get install -y osmium-tool

print("\n✓ Installation complete!")
print("Checking osmium version:")
!osmium --version

import requests
import subprocess
import os

def download_pbf(url, output_file="region.osm.pbf"):
    """
    Download PBF file from a URL (e.g., Geofabrik).

    Args:
        url: URL to download PBF from
        output_file: Name of the output file (default: region.osm.pbf)

    Returns:
        str: Path to downloaded file, or None if failed
    """

    print("=" * 70)
    print("DOWNLOADING PBF FILE")
    print("=" * 70)
    print(f"URL: {url}")
    print(f"Saving to: {output_file}\n")

    try:
        response = requests.get(url, stream=True)
        response.raise_for_status()

        total_size = int(response.headers.get('content-length', 0))
        downloaded = 0

        with open(output_file, 'wb') as f:
            for chunk in response.iter_content(chunk_size=1024*1024):  # 1MB chunks
                if chunk:
                    f.write(chunk)
                    downloaded += len(chunk)

                    if total_size > 0:
                        percent = (downloaded / total_size) * 100
                        mb_downloaded = downloaded / (1024*1024)
                        mb_total = total_size / (1024*1024)
                        print(f"\rProgress: {percent:.1f}% ({mb_downloaded:.1f} MB / {mb_total:.1f} MB)",
                              end='', flush=True)

        print(f"\n✓ Downloaded successfully: {output_file}")
        file_size = os.path.getsize(output_file) / (1024*1024)
        print(f"  File size: {file_size:.2f} MB")
        print("=" * 70)

        return output_file

    except requests.exceptions.RequestException as e:
        print(f"\n✗ Download failed: {e}")
        return None


def clip_pbf(input_pbf, minlon, minlat, maxlon, maxlat, output_file="clipped.osm.pbf"):
    """
    Clip a PBF file to a bounding box.

    Args:
        input_pbf: Path to input PBF file
        minlon, minlat, maxlon, maxlat: Bounding box coordinates
        output_file: Name of the output clipped file

    Returns:
        str: Path to clipped file, or None if failed
    """

    print("\n" + "=" * 70)
    print("CLIPPING TO BOUNDING BOX")
    print("=" * 70)
    print(f"Input file: {input_pbf}")
    print(f"Output file: {output_file}")
    print(f"\nBounding box:")
    print(f"  Min: ({minlat}, {minlon})")
    print(f"  Max: ({maxlat}, {maxlon})")
    print(f"  Area: ~{(maxlat-minlat)*111:.1f} km x {(maxlon-minlon)*111:.1f} km")

    try:
        # Check if input file exists
        if not os.path.exists(input_pbf):
            print(f"✗ Input file not found: {input_pbf}")
            return None

        # Check if osmium is installed
        check_osmium = subprocess.run(
            ['osmium', '--version'],
            capture_output=True,
            text=True
        )

        if check_osmium.returncode != 0:
            raise FileNotFoundError("osmium not found")

        # Clip the file
        result = subprocess.run([
            'osmium', 'extract',
            '--bbox', f'{minlon},{minlat},{maxlon},{maxlat}',
            '-o', output_file,
            input_pbf,
            '--overwrite'
        ], check=True, capture_output=True, text=True)

        print(f"✓ Clipping complete!")

        # Get output file size
        output_size = os.path.getsize(output_file) / (1024*1024)
        print(f"✓ Saved to: {output_file}")
        print(f"  File size: {output_size:.2f} MB")
        print("=" * 70)

        return output_file

    except FileNotFoundError:
        print("✗ osmium-tool not found!")
        return None

    except subprocess.CalledProcessError as e:
        print(f"✗ Clipping failed!")
        print(f"Error: {e.stderr}")
        return None


# =============================================================================
# USAGE EXAMPLE
# =============================================================================

# Step 1: Download region PBF (do this once, reuse for multiple clips)
region_pbf = download_pbf(
    url="https://download.geofabrik.de/asia/india-latest.osm.pbf",
    output_file="india-latest.osm.pbf"
)

if region_pbf:
    # Step 2: Clip to your bounding box
    clipped_region = clip_pbf(
        input_pbf=region_pbf,
        minlat =  18.95477,
        minlon = 84.025331,
        maxlat = 19.55550,
        maxlon =  90.0328,



        output_file="hyderabad_region.osm.pbf"
    )

    if clipped_region:
        print("\n" + "=" * 70)
        print("✅ SUCCESS!")
        print("=" * 70)
        print(f"Your clipped PBF file: {clipped_region}")
        print(f"Location: /content/{clipped_region}")
        print("=" * 70)
    else:
        print("\n❌ Clipping failed.")
else:
    print("\n❌ Download failed.")

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  libboost-program-options1.74.0
The following NEW packages will be installed:
  libboost-program-options1.74.0 osmium-tool
0 upgraded, 2 newly installed, 0 to remove and 130 not upgraded.
Need to get 882 kB of archives.
After this operation, 3,863 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu jammy/main amd64 libboost-program-options1.74.0 amd64 1.74.0-14ubuntu3 [311 kB]
Get:2 http://archive.ubuntu.com/ubuntu jammy/universe amd64 osmium-tool amd64 1.14.0-1 [571 kB]
Fetched 882 kB in 0s (2,095 kB/s)
Selecting previously unselected package libboost-program-options1.74.0:amd64.
(Reading database ... 118243 files and dire

In [ ]:
from google.colab import drive
drive.mount('/content/drive')